## TabuLa

In [1]:
from tabula.tabula import Tabula
import pandas as pd
import torch

2024-11-25 10:33:43.385952: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-25 10:33:43.946312: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
file_path = '../../Data/data.csv'
data = pd.read_csv(file_path)

In [3]:
pd.set_option('display.max_columns', None)
print(data.shape)
data.head()

(10000, 8)


,customer_age_class,region,basket_package_price_disc,internet_package_price_disc,basket_days_until_contract_end,count_previous_cancellations_0_18,internet_product_name,flag_churn
0,70 - 74,NRW,64.95,54.97,366,0,3PLAY PREMIUM HRZ,0
1,35 - 39,HSN,44.98,44.98,100,0,RED INTERNET & PHONE CABLE U,1
2,35 - 39,NRW,54.98,54.98,30,0,RED INTERNET & PHONE CABLE U,1
3,35 - 39,HSN,209.84,209.84,30,0,VF CABLEMAX,1
4,25 - 29,NRW,63.98,63.98,123,0,GIGAZUHAUSE KABEL,1


In [4]:
print(data.shape)
categorical_columns = ['customer_age_class', 'region', 'basket_product_category', 'internet_product_name']

(10000, 8)


In [22]:
colunms_internet = ['VF', 'GIGAZUHAUSE', '2PLAY', 'RED', 'EAZY', '3PLAY', 'VFORT',
       'INTERNET', '3PLAY99', 'TREUE', '3PLAYART']

In [10]:
model = Tabula(llm='distilgpt2', experiment_dir = "adult_training", batch_size=32, epochs=400, categorical_columns = categorical_columns)

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
model.fit(data)

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss
500,0.964600
1000,0.376300
1500,0.340800
2000,0.327400
2500,0.319500
3000,0.314400
3500,0.310700
4000,0.307400
4500,0.304400
5000,0.301500


 [ 3086/16000 1:15:04 < 5:14:24, 0.68 it/s, Epoch 77.12/400]

In [6]:
# Optional transforamtion for categories
import string

internet_unique = data['internet_product_name'].unique()
#baskate_unique = data['basket_product_category'].unique()

def create_mapping(unique_values, prefix):
    alphabet = string.ascii_lowercase
    codes = [f"{prefix}{a}{b}" for a in alphabet for b in alphabet]
    if len(unique_values) > len(codes):
        raise ValueError("Not enough unique codes to cover all values")
        
    encode_map = dict(zip(unique_values, codes[:len(unique_values)]))
    decode_map = {v: k for k, v in encode_map.items()}
    return encode_map, decode_map

internet_encode, internet_decode = create_mapping(internet_unique, 'i')
#basket_encode, basket_decode = create_mapping(baskate_unique, 'b')

data['internet_product_name'] = data['internet_product_name'].map(internet_encode)
#data['basket_product_category'] = data['basket_product_category'].map(internet_encode)

In [8]:
data.head(34)

,customer_age_class,region,basket_package_price_disc,internet_package_price_disc,basket_days_until_contract_end,count_previous_cancellations_0_18,internet_product_name,flag_churn
0,70 - 74,NRW,64.95,54.97,366,0,iaa,0
1,35 - 39,HSN,44.98,44.98,100,0,iab,1
2,35 - 39,NRW,54.98,54.98,30,0,iab,1
3,35 - 39,HSN,209.84,209.84,30,0,iac,1
4,25 - 29,NRW,63.98,63.98,123,0,iad,1
5,70 - 74,HSN,44.98,44.98,30,0,iab,0
6,55 - 59,HSN,34.99,34.99,93,0,iab,1
7,55 - 59,KBW,78.90,57.91,30,0,iae,0
8,55 - 59,NRW,49.98,34.99,142,0,iaf,0
9,35 - 39,NRW,0.00,0.00,30,0,iag,1


In [14]:
# distilgpt model
categorical_columns = ['customer_age_class', 'region', 'internet_product_name']
model = Tabula(llm='distilgpt2', experiment_dir = "adult_training", batch_size=64, epochs=50, categorical_columns = categorical_columns)

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [16]:
model.fit(data)

/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss


In [17]:
synthetic_data_tabula = model.sample(n_samples=10000)

10044it [01:45, 95.48it/s]                           
/home/jupyter/Students/2024_Mayilyan_LLM/LLM_based/TabuLa/tabula/tabula.py:103: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[self.label_encoder_list[i]['column']] = pd.to_numeric(data[self.label_encoder_list[i]['column']], errors='coerce')


In [19]:
print(synthetic_data_tabula.shape)
synthetic_data_tabula.head()

(9970, 8)


,customer_age_class,region,basket_package_price_disc,internet_package_price_disc,basket_days_until_contract_end,count_previous_cancellations_0_18,internet_product_name,flag_churn
0,40 - 44,NRW,37.99,34.99,30.0,0.0,VF CABLEMAX,0.0
1,25 - 29,NRW,39.99,44.98,30.0,0.0,RED INTERNET & PHONE CABLE U,1.0
2,70 - 74,NRW,39.98,39.97,30.0,0.0,RED INTERNET & PHONE CABLE U,0.0
3,50 - 54,NRW,44.99,44.98,30.0,0.0,GIGAZUHAUSE KABEL,1.0
4,35 - 39,NRW,44.98,44.99,30.0,0.0,VF RIP U-TREUE,0.0
...,...,...,...,...,...,...,...,...
9995,45 - 49,NRW,39.99,39.99,30.0,0.0,VF RIP U-TREUE,0.0
9996,55 - 59,HSN,39.99,34.99,30.0,0.0,GIGAZUHAUSE KABEL,1.0
9997,40 - 44,NRW,29.98,49.98,30.0,0.0,RED INTERNET & PHONE CABLE U,1.0
9998,80 and over,HSN,38.97,44.99,30.0,0.0,RED INTERNET & PHONE CABLE U,1.0


In [20]:
synthetic_data_tabula.to_csv('tabula_data.csv')

In [13]:
# gpt model
categorical_columns = ['customer_age_class', 'region', 'internet_product_name']
model_gpt = Tabula(llm='gpt2', experiment_dir = "adult_training", batch_size=32, epochs=5, categorical_columns = categorical_columns)

In [14]:
model_gpt.fit(data)

/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss


In [15]:
synthetic_data_tabula_gpt = model_gpt.sample(n_samples=10000)

10045it [02:51, 58.56it/s]                          
/home/jupyter/Students/2024_Mayilyan_LLM/LLM_based/TabuLa/tabula/tabula.py:103: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[self.label_encoder_list[i]['column']] = pd.to_numeric(data[self.label_encoder_list[i]['column']], errors='coerce')
/home/jupyter/Students/2024_Mayilyan_LLM/LLM_based/TabuLa/tabula/tabula.py:107: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[self.label_encoder_list[i]['column']] = data[self.label_encoder_list[i]['column']]

In [17]:
print(synthetic_data_tabula_gpt.shape)
synthetic_data_tabula_gpt.head()

(9956, 8)


,customer_age_class,region,basket_package_price_disc,internet_package_price_disc,basket_days_until_contract_end,count_previous_cancellations_0_18,internet_product_name,flag_churn
0,60 - 64,KBW,39.99,37.99,30.0,0.0,RED INTERNET & PHONE CABLE U,0.0
1,35 - 39,NRW,49.97,44.98,30.0,0.0,RED INTERNET & PHONE CABLE U,1.0
2,60 - 64,KBW,37.97,37.97,30.0,0.0,VF CABLEMAX,0.0
3,25 - 29,HSN,37.98,29.99,30.0,0.0,RED INTERNET & PHONE CABLE U,1.0
4,40 - 44,KBW,44.98,44.98,30.0,0.0,VF CABLEMAX,0.0


In [18]:
synthetic_data_tabula_gpt.to_csv('tabula_data_gpt.csv')

## TabuLa with middle Padding

In [31]:
from tabula_middle_padding.tabula import Tabula
import pandas as pd
import torch

In [32]:
file_path = '../../Data/data.csv'
data = pd.read_csv(file_path)

In [33]:
pd.set_option('display.max_columns', None)
print(data.shape)

columns = ['region', 'customer_age_class', 'basket_package_price_disc', 'internet_package_price_disc', 
           'basket_days_until_contract_end', 'count_previous_cancellations_0_18', 'internet_product_name','flag_churn']
data = data[columns]
data.head()

(10000, 8)


,region,customer_age_class,basket_package_price_disc,internet_package_price_disc,basket_days_until_contract_end,count_previous_cancellations_0_18,internet_product_name,flag_churn
0,NRW,70 - 74,64.95,54.97,366,0,3PLAY PREMIUM HRZ,0
1,HSN,35 - 39,44.98,44.98,100,0,RED INTERNET & PHONE CABLE U,1
2,NRW,35 - 39,54.98,54.98,30,0,RED INTERNET & PHONE CABLE U,1
3,HSN,35 - 39,209.84,209.84,30,0,VF CABLEMAX,1
4,NRW,25 - 29,63.98,63.98,123,0,GIGAZUHAUSE KABEL,1


In [34]:
categorical_columns = ['customer_age_class', 'region', 'internet_product_name']

In [21]:
# distil gpt model
model_tabula_middle = Tabula(llm='distilgpt2', experiment_dir = "adult_training", batch_size=32, epochs=10, categorical_columns=categorical_columns)

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [23]:
model_tabula_middle.fit(data, conditional_col = 'region')

/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss


In [24]:
synthetic_data_tabula_middle = model_tabula_middle.sample(n_samples=10000)

10096it [01:44, 96.88it/s]                          


In [25]:
print(synthetic_data_tabula_middle.shape)
synthetic_data_tabula_middle.head()

(10000, 8)


,region,customer_age_class,basket_package_price_disc,internet_package_price_disc,basket_days_until_contract_end,count_previous_cancellations_0_18,internet_product_name,flag_churn
0,KBW,40 - 44,42.98,42.98,30.0,0.0,VF RIP U-TREUE,1.0
1,NRW,35 - 39,39.98,39.98,30.0,0.0,RED INTERNET & PHONE CABLE U,1.0
2,NRW,25 - 29,39.98,39.98,84.0,0.0,VF CABLEMAX,1.0
3,NRW,30 - 34,48.97,48.97,30.0,0.0,VF CABLEMAX,1.0
4,KBW,25 - 29,44.98,44.98,30.0,0.0,RED INTERNET & PHONE CABLE U,0.0


In [27]:
synthetic_data_tabula_middle.to_csv("tabula_data_middle_padding.csv", index=False)

In [35]:
# gpt model
model_tabula_middle_gpt = Tabula(llm='gpt2', experiment_dir = "adult_training", batch_size=32, epochs=10, categorical_columns=categorical_columns)

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [36]:
model_tabula_middle_gpt.fit(data, conditional_col = 'region')

/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss


In [37]:
synthetic_data_tabula_middle_gpt = model_tabula_middle_gpt.sample(n_samples=10000)

10092it [02:52, 58.40it/s]                          


In [39]:
print(synthetic_data_tabula_middle_gpt.shape)
synthetic_data_tabula_middle_gpt.head()

(10000, 8)


,region,customer_age_class,basket_package_price_disc,internet_package_price_disc,basket_days_until_contract_end,count_previous_cancellations_0_18,internet_product_name,flag_churn
0,KBW,35 - 39,37.99,37.99,30.0,0.0,VF RIP U-TREUE,0.0
1,NRW,30 - 34,44.98,44.98,30.0,0.0,VF CABLEMAX,1.0
2,NRW,45 - 49,49.99,49.99,30.0,0.0,RED INTERNET & PHONE CABLE U,0.0
3,NRW,65 - 69,39.98,39.98,30.0,0.0,RED INTERNET & PHONE CABLE U,1.0
4,KBW,50 - 54,52.96,52.96,30.0,0.0,VF CABLEMAX,0.0


In [40]:
synthetic_data_tabula_middle_gpt.to_csv("tabula_data_middle_padding_gpt.csv", index=False)

## Fine-tuned

In [1]:
from tabula.tabula import Tabula
import pandas as pd
import torch

file_path = '../../Data/data.csv'
data = pd.read_csv(file_path)

print(data.shape)

columns = ['region', 'customer_age_class', 'basket_package_price_disc', 'internet_package_price_disc', 
           'basket_days_until_contract_end', 'count_previous_cancellations_0_18', 'internet_product_name','flag_churn']
data = data[columns]
# distilgpt model
categorical_columns = ['customer_age_class', 'region', 'internet_product_name']

2024-11-16 15:36:16.050454: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-16 15:36:16.113636: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


(10000, 8)


In [ ]:
# tabula-distilgpt
model_tabula_v2 = Tabula(llm='distilgpt2', experiment_dir = "adult_training", batch_size=32, epochs=50, categorical_columns = categorical_columns)

model_tabula_v2.fit(data)

synthetic_data_tabula_v2 = model_tabula_v2.sample(n_samples=10000)
synthetic_data_tabula_v2.head()

synthetic_data_tabula_v2.to_csv('tabula_data_v2.csv')

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsq

Step,Training Loss
500,0.781200
1000,0.357000
1500,0.340300
2000,0.333900


10078it [01:08, 147.31it/s]                          
/home/jupyter/Students/2024_Mayilyan_LLM/LLM_based/TabuLa/tabula/tabula.py:103: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[self.label_encoder_list[i]['column']] = pd.to_numeric(data[self.label_encoder_list[i]['column']], errors='coerce')


In [2]:
# tabula-gpt
model_tabula_gpt_v2 = Tabula(llm='gpt2', experiment_dir = "adult_training", batch_size=32, epochs=50, categorical_columns = categorical_columns)

model_tabula_gpt_v2.fit(data)

synthetic_data_tabula_gpt_v2 = model_tabula_gpt_v2.sample(n_samples=10000)
synthetic_data_tabula_gpt_v2.head()

synthetic_data_tabula_gpt_v2.to_csv('tabula_data_gpt_v2.csv')

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsq

Step,Training Loss
500,0.792200
1000,0.353600
1500,0.337200
2000,0.330300


10084it [01:49, 91.71it/s]                          
/home/jupyter/Students/2024_Mayilyan_LLM/LLM_based/TabuLa/tabula/tabula.py:103: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[self.label_encoder_list[i]['column']] = pd.to_numeric(data[self.label_encoder_list[i]['column']], errors='coerce')


In [2]:
# tabula-middle-distilgpt
model_tabula_middle_v2 = Tabula(llm='distilgpt2', experiment_dir = "adult_training", batch_size=32, epochs=25)

model_tabula_middle_v2.fit(data, conditional_col = 'region')

synthetic_data_tabula_middle_v2 = model_tabula_middle_v2.sample(n_samples=10000)

synthetic_data_tabula_middle_v2.to_csv('tabula_data_middle_v2.csv', index=False)
synthetic_data_tabula_middle_v2.head()

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsq

Step,Training Loss
500,0.800800
1000,0.318200


10049it [01:20, 124.97it/s]                          


,region,customer_age_class,basket_package_price_disc,internet_package_price_disc,basket_days_until_contract_end,count_previous_cancellations_0_18,internet_product_name,flag_churn
0,KBW,30,42.98,42.98,30.0,0.0,GIGAZUHAUSE,0.0
1,NRW,40,54.98,54.99,30.0,0.0,GIGAZUHAUSE,0.0
2,HSN,45,69.96,44.98,30.0,0.0,RED,1.0
3,HSN,25,24.99,24.99,277.0,0.0,GIGAZUHAUSE,0.0
4,NRW,40,39.99,39.99,30.0,0.0,RED,1.0


In [36]:
#internet name columns was generated with different values
import numpy as np

column_internet_data = data['internet_product_name'].unique()
columns_internet_synthetic = synthetic_data_tabula_middle_v2['internet_product_name'].unique()

max_length = len(column_internet_data)
tabula_model_padded = np.pad(columns_internet_synthetic, (0, max_length - len(columns_internet_synthetic)), constant_values=None)

df = pd.DataFrame({'Original Data': column_internet_data,
                  'Synthetic': tabula_model_padded})
df.to_csv('../../Data/internet_columns.csv', index=False)

In [14]:
# made the '35' string into an iterval '35-39'
synthetic_data_tabula_middle_v2['customer_age_class'] = pd.to_numeric(synthetic_data_tabula_middle_v2['customer_age_class'], errors='coerce')

bins = [-float('inf'), 17, 24, 29, 34, 39, 44, 49, 54, 59, 64, 69, 74, 79, float('inf')]
labels = ['70 - 74', '35 - 39', '25 - 29', '55 - 59', '75 - 79', 'under 18',
       '65 - 69', '60 - 64', '50 - 54', '45 - 49', '40 - 44', '30 - 34',
       '18 - 24', '80 and over']

synthetic_data_tabula_middle_v2['customer_age_class'] = pd.cut(synthetic_data_tabula_middle_v2['customer_age_class'], bins=bins, labels=labels, right=True)
synthetic_data_tabula_middle_v2['customer_age_class'] = synthetic_data_tabula_middle_v2['customer_age_class'].cat.add_categories(['not available']).fillna('not available')

In [8]:
# tabula-middle-gpt
model_tabula_middle_gpt_v2 = Tabula(llm='gpt2', experiment_dir = "adult_training", batch_size=16, epochs=20, categorical_columns = categorical_columns)

model_tabula_middle_gpt_v2.fit(data, conditional_col = 'region')

synthetic_data_tabula_middle_gpt_v2 = model_tabula_middle_gpt_v2.sample(n_samples=10000)

synthetic_data_tabula_middle_gpt_v2.to_csv('tabula_data_middle_gpt_v2.csv', index=False)
synthetic_data_tabula_middle_gpt_v2.head()

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsq

Step,Training Loss
500,0.803100
1000,0.361400
1500,0.344000


10087it [01:54, 87.80it/s]                          
/home/jupyter/Students/2024_Mayilyan_LLM/LLM_based/TabuLa/tabula/tabula.py:103: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[self.label_encoder_list[i]['column']] = pd.to_numeric(data[self.label_encoder_list[i]['column']], errors='coerce')


,region,customer_age_class,basket_package_price_disc,internet_package_price_disc,basket_days_until_contract_end,count_previous_cancellations_0_18,internet_product_name,flag_churn
0,2,7,34.99,34.99,30.0,0.0,46,0.0
1,2,9,44.98,44.98,30.0,0.0,51,1.0
2,1,13,34.99,34.99,30.0,0.0,46,1.0
3,2,2,44.97,44.97,30.0,0.0,51,0.0
4,2,1,34.99,34.99,30.0,0.0,46,1.0


In [6]:
synthetic_data_tabula_middle_gpt_v2 = model_tabula_middle_gpt_v2.sample(n_samples=100)

synthetic_data_tabula_middle_gpt_v2.head()

100%|██████████| 100/100 [00:01<00:00, 92.16it/s]


,region,customer_age_class,basket_package_price_disc,internet_package_price_disc,basket_days_until_contract_end,count_previous_cancellations_0_18,internet_product_name,flag_churn
0,1,6,49.99,49.99,30.0,0.0,50,1.0
1,1,6,44.98,44.98,30.0,0.0,55,1.0
2,2,3,39.98,39.98,30.0,0.0,38,1.0
3,2,10,39.99,39.99,30.0,0.0,50,0.0
4,2,11,49.99,49.99,30.0,0.0,50,0.0


In [7]:
data.head()

,region,customer_age_class,basket_package_price_disc,internet_package_price_disc,basket_days_until_contract_end,count_previous_cancellations_0_18,internet_product_name,flag_churn
0,2,2,64.95,54.97,366,0,17,0
1,0,8,44.98,44.98,100,0,46,1
2,2,8,54.98,54.98,30,0,46,1
3,0,8,209.84,209.84,30,0,51,1
4,2,1,63.98,63.98,123,0,32,1
